## Initial Design

main components for course recommendation system

1. Eligibility_pipeline - Based on the student academic records such as school, competetive exams, UG / PG records etc.. data will be used to check eligibility from the courses that university provides.

2. Similarity_search_model - cosine similarity engine to search for relevant courses among the eligible courses. Then rank the Top-K courses as the output


## Current action plan

First the course data will be required in the structured format to store it as vector embedding, which will be used for similarity search.

Course data will be structured as follows:

```python
1. course_id
2. course_name
3. degree_level        # UG / PG / PhD
4. domain              # Engineering / Management /5. Science / Humanities

eligible_degrees    # list-like string
required_subjects   # list-like string

required_exam       # JEE / CAT / GATE / NET / NONE
min_exam_score
min_gpa

description         # for TF-IDF
keywords            # comma separated
career_outcomes     # optional but useful
```


## First Implementation

The first code implemetation originated from the need of a clean dataset for the eligibility pipeline. 

The current dataset had the following fields.
1. Program Name
2. Program level
3. Name of Faculty
4. Eligibility criteria

There were more fields in the datset but for the project's scope these are relevant.

The restructuring pipeline [restructure.ipynb](../../restructure_pipeline/restructure.ipynb) has the complete implementation.
The pipeline is a small Agentic workflow made with langchain-ollama library uses a locally running model `llama3.2` to generate a structure output which then will be handled by the `pandas` library to generate the desired dataset.

The ouput dataset will have the following fields
1. program_name: str 
2. program_level: str 
3. domain: str 
4. eligibility: str 
5. description: str 
6. skills_learned: List[str]
7. career_outcomes: List[str] 

The structure output is genrate by the model using pydantic model to ensure type safety. Dateset with these dimensions will be used for eligibility criteria and ML similarity to generate recommendations.

## Eligibility Pipeline

The dataset has been finalised and is ready for eligibility pipeline. This pipeline will serve the purpose of filteriing all the eligible course for a student profile. The filtered courses then will be matched against the student's profile in a cosine similarity engine.

The finalised dataset has the following schema:-
```python
class EligibilityStruct(BaseModel):
    min_degree_level: Optional[str] = Field(
        default=None,
        description="One of: PreUG, UG, PG, PhD"
    )
    min_marks_general: Optional[float] = Field(
        default=None,
        description="Minimum percentage for general category",
        ge=0.0,
        le=100.0
    )
    min_marks_reserved: Optional[float] = Field(
        default=None,
        description="Minimum percentage for reserved category (SC/ST/OBC/PwD)",
        ge=0.0,
        le=100.0
    )

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )
    eligibility_struct: EligibilityStruct = Field(description="Eligibiliy in the structured format")
```

The dataset schema is validates using pydantic models ensuring type-safety during runtime.

## Data Flattening script

This script was used for flattening the `eligibility_struct` field as it was causing parsing errors in the eligibility pipeline.

In [8]:
import pandas as pd
import re

def parse_eligibility_struct(text):
    degree = re.search(r"min_degree_level='(.*?)'", text)
    gen = re.search(r"min_marks_general=([\d.]+)", text)
    res = re.search(r"min_marks_reserved=([\d.]+)", text)

    return pd.Series({
        "min_degree_level": degree.group(1) if degree else None,
        "min_marks_general": float(gen.group(1)) if gen else None,
        "min_marks_reserved": float(res.group(1)) if res else None
    })
df = pd.read_csv("final_data2.csv")

elig_cols = df["eligibility_struct"].apply(parse_eligibility_struct)

df = pd.concat([df, elig_cols], axis=1, names="struct_eligibility")
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0", "eligibility_struct"])
df.to_csv("final_data3.csv", index=False)
print(f"flattened {len(df)} records")


    


flattened 114 records


### Experiment Log: TF-IDF Based Course Recommendation System

#### Objective

To evaluate the effectiveness of TF-IDF vectorization combined with cosine similarity for recommending academic courses based on student profiles.

---

#### Implementation Summary

* Course features used:

  * Program Name
  * Domain
  * Description
  * Skills Learned
  * Career Outcomes

* Student features used:

  * Academic Background
  * Subjects
  * Interests
  * Preferred Skills
  * Career Goal
  * Preferred Domain

* Pipeline:

  1. TF-IDF vectorizer fitted on all course texts
  2. Student and eligible course texts transformed into vectors
  3. Cosine similarity used to rank courses
  4. Top-K courses selected as recommendations

---

#### Observations

1. **Top-K Quality Distribution**

   * The top 1–2 recommendations are highly relevant
   * Top 3–5 are moderately relevant
   * Beyond top 5, relevance drops significantly

2. **Domain Alignment**

   * Highest-ranked recommendations often align with the student's preferred domain
   * Secondary recommendations align more with skills/interests rather than domain

3. **Quality Degradation Pattern**

   * Ranking quality decreases progressively
   * Lower-ranked results include semantically weak matches or domain mismatches

---

#### Root Cause Analysis

1. **TF-IDF Limitation**

   * Relies on keyword overlap rather than semantic meaning
   * Cannot distinguish contextual differences (e.g., "data analysis" in biology vs computer science)

2. **Feature Imbalance**

   * Long descriptions dominate vector representation
   * Domain signal is underweighted

3. **Dataset Constraints**

   * Limited diversity of courses
   * Overlap of generic keywords across domains introduces noise

---

#### Key Insights

* TF-IDF is effective for **high-confidence top recommendations (Top-1 to Top-3)**
* It fails in **fine-grained ranking beyond top candidates**
* Domain awareness is weak without explicit weighting or filtering
* Suitable as a **baseline model**, not a final solution

---

#### Conclusion

TF-IDF with cosine similarity provides a strong baseline for course recommendation systems, especially for identifying top candidates. However, it lacks semantic understanding and domain sensitivity, leading to reduced effectiveness in deeper ranking.

---

#### Next Direction

* Introduce embedding-based similarity for semantic understanding
* Compare TF-IDF vs embedding performance on ranking quality
* Explore hybrid approaches combining both methods
* Implement evaluation metrics (e.g., Precision@K, domain match rate)


## Experiment Log: Common Vector engine Interface

This is a project journal log for the 'course recommender engine'. In this entry, the initialisation of common vector engine will be discussed.

---

### Intuition

The need for the common vector engine arises from the base of this project i.e to compare different algorithms and techniques for the recommender engine. The comparision process was becoming a mess when it comes to code implementation becuase the process includes the following:- 
1. The bulk data vectorisation (courses).
2. The iteration over every student for every course.
3. vectorising the student profile as a query.
4. Calculation the cosine_similarity.
5. Record and store results structurally.

The process was becoming a repeated task but required changes everytime a different algorithm is tested. Therefore a common a interface was initialised

---

### Implementation Specifics

The implementation was quite structured with keeping in mind a common interface is required for all different techniques. Therefore a common class `VectorEngine` is created with three key function that define the entire process of vectorisation and cosine similarity `.fit()` , `.transform()`, `.similarity()` .

```python
class VectorEngine:

    def fit(self, corpus: list[str]):
        pass

    def transform(self, texts: list[str]):
        pass

    def similarity(self, query_vec, matrix):
        pass
```

The implementation in the interface was left deliberately empty to provide just the interface, the algorithm classes can inherit and create implementation specific to the algorithm. All 3 techniques now use this interface in implementation, for eg - EmbeddingEngine implementation looks like - 
```python
class EmbeddingEngine(VectorEngine):
    
    def __init__(self):
        self.model_name = "nomic-embed-text"
        self.model_url = "http://127.0.0.1:11434/api/embeddings"
        self.matrix = None

    def generate_embeddings(self, entity) -> list[float]:
        response = requests.post(url=self.model_url ,json={
            "model":self.model_name,
            "prompt": entity
        })

        if response.status_code != 200:
            raise Exception(f"Embedding API failed: {response.text}")

        data = response.json()

        return data["embedding"]


    def fit(self, corpus):
        self.matrix = np.array([
            self.generate_embeddings(text)
            for text in corpus
        ])
    
    def transform(self, texts):
        return np.array([
            self.generate_embeddings(text)
            for text in texts
        ])
    
    def similarity(self, query_vec, matrix=None):

        matrix = matrix if matrix is not None else self.matrix
        dot = np.dot(matrix, query_vec.T).squeeze()
        norm = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vec)
        return dot / norm
```

This implementation follows the interface from the `VectorEngine` class. This interface and implementation will provide a quick way to create production implementaion for the recommender engine even using more that 1 algorithm.


# Log: Streamlit Frontend & Test observation

This log refers to the development of the fronend module to test the recommendation engines and acknowledge the observations of the result.

---

A streamlit based frontend application has been developed for the gathering of recommendation results from university students for the purpose of training ML algorithms.
The main file for this application is `app.py` and the results are currenlty being stored in the local database created using sqlite3 and managed using `sqlalchemy` ORM.

There few changes in the app to enhance the recommendations and are subject to the extension of the courses data. Two new fields have been included from the data to recommendation corpus data, these are:-
1. `mode` - Mode of studying the course 'Regular', 'Distance' or 'Part time'
2. `Duraion` - Courses are of varied duration, students can opt for their preferred duration.

email_address = sa@mdu.ac.in

In [5]:
import pandas as pd
import numpy as np
import ast

In [ ]:
def transform(x):
    if x == 'ANY':
        return '[]'
    return x
    
    
df["required_subjects"] = df["required_subjects"].apply(transform)

print(df[features].head())


In [ ]:
print(df[""].value_counts())

In [ ]:
import ast
course_df = pd.read_csv("restructure_pipeline/fixed_data.csv")

course_df['skills_learned'] = course_df['skills_learned'].apply(ast.literal_eval)
course_df['career_outcomes'] = course_df['career_outcomes'].apply(ast.literal_eval)
print(course_df['skills_learned'].head())
print(course_df['career_outcomes'].head())


In [62]:
course_df.to_csv("courses_data.csv", index=False)


In [83]:
df = pd.read_csv("dataset/courses_data.csv")
df['required_subjects'] = df['required_subjects'].apply(ast.literal_eval)
df['career_outcomes'] = df['career_outcomes'].apply(ast.literal_eval)
df['skills_learned'] = df['skills_learned'].apply(ast.literal_eval)

print(f"{df['required_subjects'].dtype} {df['career_outcomes'].dtype} {df['skills_learned'].dtype}")

df.to_csv("dataset/courses_data.csv")

object object object


In [78]:
from recommender_system import EmbeddingEngine

embed_engine  = EmbeddingEngine()
student_vec = embed_engine.transform([" ".join(["Environmental Assessment", "Data Analysis", "Field Research"])])
course_vec = embed_engine.transform([" ".join(df.iloc[10]['skills_learned'])])
skill_relevance_percentage = embed_engine.similarity(student_vec, course_vec) * 100
print(skill_relevance_percentage)

[61.00185714]


In [ ]:
import sqlite3
import pandas as pd
import json


def load_training_dataset():

    conn=sqlite3.connect(
        "experiment.db"
    )

    df=pd.read_sql(
        "SELECT * FROM dataset",
        conn
    )

    conn.close()

    json_cols=[

    "subjects",
    "interests",
    "preferred_skills",

    "skills_learned",

    "career_outcomes",

    "required_subjects"

    ]

    for col in json_cols:

        df[col] = df[col].apply(lambda x: [] if pd.isna(x) else ast.literal_eval(x))




    bool_cols=[

    "is_reserved",

    "domain_match",

    "subject_required",

    "marks_required"

    ]

    for col in bool_cols:

        df[col]=df[col].astype(bool)


    numeric_cols=[

    "percentage",

    "similarity_score",

    "subject_overlap",

    "marks_margin",

    "skill_relevance_percentage",

    "career_align_percentage",

    "label"

    ]

    for col in numeric_cols:

        df[col]=pd.to_numeric(
            df[col],
            errors="coerce"
        )
    df['min_marks_general'] = (
        df["min_marks_general"].fillna(-1)
    )
    df['min_marks_reserved'] = (
        df["min_marks_reserved"].fillna(-1)
    )
    df['marks_margin'] = np.where(
        df['marks_required'] == 0,
        -1,
        df['marks_margin']
    )
    df['duration'] = (
        df['duration'].fillna("N/A")
    )
    return df

dataset = load_training_dataset()
print(dataset['required_subjects'][200])
print(dataset['required_subjects'].dtype)
print(dataset['is_reserved'].dtype)
print(dataset['required_subjects'].isna().sum())
print(dataset.isna().sum())

In [ ]:
from sklearn.feature_selection import mutual_info_regression

dataset = load_training_dataset()
features = [
    "similarity_score",
    "domain_match",
    "subject_required",
    "subject_overlap",
    "marks_required",
    "marks_margin",
    "skill_relevance_percentage",
    "career_align_percentage",
    "label"
]
X = dataset[features].copy()
y = dataset[features].pop('label')


def make_scores(X, y):
    scores = mutual_info_regression(X, y)
    scores = pd.Series(scores, name="mi_scores", index=X.columns)
    scores = scores.sort_values(ascending=False)
    return scores

scores = make_scores(X, y)
print(scores)

# Log: First Iteration of Machine Learning

---

This log records the result of the first iteration and results of the Machine Learning algorithm for course recommender system. 

Before the ML algorithm was feed with the features, some data transformations had to be made since the data was stored in a SQL database `sqlite3` for now. Some features had numerous NaN values like - `required_subjects`, `marks_margin`, `subjects_required` but a python function helped to fill the NaN values with appropriate data type values.

For `required_subjects` NaN represented an empty array so it was replaced with an empty array. `marks_margin` was a numeric value therefore -1 was entered in place of NaN to represent absence of a real value instead of zero which would have indicated that cutoff was barely met.

This script was used to transform the data for training...

```python
import sqlite3
import pandas as pd
import numpy as np
import ast


def load_training_dataset():

    conn=sqlite3.connect(
        "experiment.db"
    )

    df=pd.read_sql(
        "SELECT * FROM dataset",
        conn
    )

    conn.close()

    json_cols=[

    "subjects",
    "interests",
    "preferred_skills",

    "skills_learned",

    "career_outcomes",

    "required_subjects"

    ]

    for col in json_cols:

        df[col] = df[col].apply(lambda x: [] if pd.isna(x) else ast.literal_eval(x))




    bool_cols=[

    "is_reserved",

    "domain_match",

    "subject_required",

    "marks_required"

    ]

    for col in bool_cols:

        df[col]=df[col].astype(bool)


    numeric_cols=[

    "percentage",

    "similarity_score",

    "subject_overlap",

    "marks_margin",

    "skill_relevance_percentage",

    "career_align_percentage",

    "label"

    ]

    for col in numeric_cols:

        df[col]=pd.to_numeric(
            df[col],
            errors="coerce"
        )
    df['min_marks_general'] = (
        df["min_marks_general"].fillna(-1)
    )
    df['min_marks_reserved'] = (
        df["min_marks_reserved"].fillna(-1)
    )
    df['marks_margin'] = np.where(
        df['marks_required'] == 0,
        -1,
        df['marks_margin']
    )
    df['duration'] = (
        df['duration'].fillna("N/A")
    )
    return df
```

After transforming the data, it was ready for the first iteration. Other important note - not all features were used for the training. Only engineered features were used, that includes:-
1. `domain_match`
2. `subject_overlap`
3. `marks_margin`
4. `career_align_percentage`
5. `skill_relevance_percentage`
6. `similarity_score`

Code in the cell below was used for the ML training and the metrics to evaluate the model performance were - **Accuracy**, **f1_score**, **classification_report**, **confusion_matrix** and **feature_importance**

These are the results from the first iteration - 
```txt
**Accuracy**: 
0.625

**F1 scores**:
0.6282568122433744

**Classification Report**: 
              precision    recall  f1-score   support

           0       0.58      0.69      0.63        26
           1       0.62      0.50      0.55        26
           2       0.70      0.70      0.70        20

    accuracy                           0.62        72
   macro avg       0.63      0.63      0.63        72
weighted avg       0.63      0.62      0.62        72


Feature Importances
                      feature     score
3             subject_overlap  0.016420
2            subject_required  0.025873
4              marks_required  0.041848
1                domain_match  0.049246
5                marks_margin  0.125924
7     career_align_percentage  0.163603
6  skill_relevance_percentage  0.167303
0            similarity_score  0.409784

confusion matrix:

18 5  3

10 13 3

3  3  14
```

These results show that model acheived an accuracy of around 62.5%, also the similarity in the number of accuracy and f1_score indicates that there is very little imbalance. 

Confusion matrix reveals something interesting about the results, although model did good on finding the `good - 2` and `bad - 0` recommendations but it is getting confused about the `moderate - 1` ones and generally classifying them as bad ones which is pessimistic. Model also picked upon the importance of `career_align_percentage` and `skill_relevance_percentage` which makes upto around 33% in feature importance but it still dominated by `similarity_score`

These results were quite good considering this is the first iteration and only engineered features were used. The plan for that next iteration is to include some categorical features from student and course profile with hot-encoding

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_selection import mutual_info_regression
from utility import load_training_dataset
import matplotlib.pyplot as plt
import pandas as pd


# Load the dataset
dataset = load_training_dataset()
features = [

    "similarity_score",

    "domain_match",

    "subject_required",

    "subject_overlap",

    "marks_required",

    "marks_margin",

    "skill_relevance_percentage",

    "career_align_percentage"

]
y = dataset.pop("label")
X = dataset[features].copy()

# Performing train, test split on the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

# Initialising and training model
model = RandomForestClassifier(random_state=42,n_estimators=200, max_depth=6)
model.fit(X_train, y_train)

# Predections on test data
predictions = model.predict(X_test)

# Metrics

print("\nAccuracy: \n")
print(accuracy_score(y_test, predictions))

print("\nF1 scores:\n")
print(f1_score(y_test, predictions, average="macro"))

print("\nClassification Report: \n")
print(classification_report(y_test, predictions))

# Metrics on features

importance = pd.DataFrame({
    "feature" : features,

    "score": model.feature_importances_ 
})

importance = importance.sort_values("score")
print(importance)

plt.barh(
    importance['feature'],
    importance['score'],
)
plt.xlabel("Importance")
plt.title("Feature Importance")
plt.show()

# Confustion Matrix

cm = confusion_matrix(y_test, predictions)
print(cm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot()
plt.show()